# CacauFito — Treino do classificador de folhas de cacau (Kaggle Notebook)

PoC de visão computacional para identificar condição fitossanitária de folhas de cacau
(sadia / cssvd / antracnose), a partir do **Amini Cocoa Contamination Dataset** (Kaggle,
licença CC BY 4.0).

**Como usar (recomendado: "Save & Run All", não rodar célula por célula):**
1. No editor do Kaggle, confirme nas configurações da sessão (painel direito):
   - **Accelerator:** GPU (T4 x2 ou P100)
   - **Internet:** ON (necessário para baixar o dataset via API do Kaggle)
2. Configure os *Secrets* do Kaggle (Add-ons > Secrets) com `KAGGLE_USERNAME` e `KAGGLE_KEY`
   (gerados em kaggle.com > Account > API > Create New Token) — a célula de autenticação já lê
   esses secrets automaticamente, sem upload manual de arquivo.
3. Clique em **Save Version > Save & Run All (Commit)**. Isso roda o notebook inteiro do início
   ao fim em background, sem depender de você manter a aba aberta — evita perder o treino
   (~60 min, incluindo ~5 min de busca de hiperparâmetros com Optuna) se a sessão interativa cair.
4. Quando o commit terminar, abra a versão salva, vá na aba **Output** e baixe
   `cacao_leaf_model_bundle.zip` (gerado na última célula).
5. Coloque os 3 arquivos do zip (`cacao_leaf_classifier.pt`, `label_mapping.json`,
   `eval_report.json`) na pasta `models/` do projeto local para o serviço de inferência usar.

**Se preferir rodar célula por célula (modo interativo):** funciona igual, mas se a sessão
cair no meio (fechar a aba, timeout de inatividade), a variável `model` em memória se perde e
é preciso rodar tudo de novo desde a célula 1 — não dá para retomar só a partir da seção 8.
"Save & Run All" evita esse problema.


## 1. Setup do ambiente

In [1]:
!pip -q install kaggle scikit-learn optuna
import torch
print("Torch:", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())


Torch: 2.10.0+cu128 | CUDA disponível: True


### 1.1 Autenticação no Kaggle

Envie o arquivo `kaggle.json` (baixado da sua conta Kaggle: Account > API > Create New Token).


In [2]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Define as variáveis de ambiente necessárias para a CLI/API do Kaggle
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")

print("Credenciais do Kaggle configuradas via Secrets.")


Credenciais do Kaggle configuradas via Secrets.


## 2. Download do dataset (Amini Cocoa Contamination Dataset, CC BY 4.0)

In [3]:
!kaggle datasets download ohagwucollinspatrick/amini-cocoa-contamination-dataset -p /content/amini --force
!cd /content/amini && unzip -q -o amini-cocoa-contamination-dataset.zip -d .
!ls /content/amini/dataset/images/train | wc -l
!ls /content/amini/dataset/images/test | wc -l


Dataset URL: https://www.kaggle.com/datasets/ohagwucollinspatrick/amini-cocoa-contamination-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100%|███████████████████████████████████████| 9.43G/9.43G [01:36<00:00, 105MB/s]

5529
1626


## 3. Montagem do manifesto (rótulo por imagem) e split treino/val/teste

O `Train.csv` original traz anotações por *bounding box* (múltiplas por imagem em alguns casos).
Como apenas 6 das 5.529 imagens têm mais de uma classe distinta, cada imagem recebe um único
rótulo (classe majoritária). O `Test.csv` oficial não tem gabarito (é o conjunto de submissão
de uma competição Zindi/Amini), então criamos nosso próprio split de teste a partir do `Train.csv`.


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
VAL_FRAC = 0.15
TEST_FRAC = 0.15
DATA_ROOT = "/content/amini"

df = pd.read_csv(f"{DATA_ROOT}/Train.csv")

def majority_class(s):
    return s.value_counts().idxmax()

manifest = (
    df.groupby("Image_ID")
    .agg(label=("class", majority_class), ImagePath=("ImagePath", "first"), num_boxes=("class", "size"))
    .reset_index()
)
manifest["source"] = "amini-cocoa-contamination-dataset"
manifest["license"] = "CC BY 4.0"

print("Total de imagens:", len(manifest))
print(manifest["label"].value_counts())

mixed = df.groupby("Image_ID")["class"].nunique()
mixed_ids = mixed[mixed > 1].index.tolist()
print(f"Imagens com classes mistas (resolvidas por maioria): {len(mixed_ids)}", mixed_ids)

train_val, test = train_test_split(
    manifest, test_size=TEST_FRAC, stratify=manifest["label"], random_state=SEED
)
train, val = train_test_split(
    train_val, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=train_val["label"], random_state=SEED
)
train["split"] = "train"
val["split"] = "val"
test["split"] = "test"
full = pd.concat([train, val, test], ignore_index=True)

# Verificação de disjunção entre os splits
assert len(set(train["Image_ID"]) & set(val["Image_ID"])) == 0
assert len(set(train["Image_ID"]) & set(test["Image_ID"])) == 0
assert len(set(val["Image_ID"]) & set(test["Image_ID"])) == 0
assert len(full) == len(manifest)

full.to_csv(f"{DATA_ROOT}/manifest.csv", index=False)
print("\nContagens por split e classe:")
print(full.groupby(["split", "label"]).size().unstack(fill_value=0))


Total de imagens: 5529
label
cssvd          2237
healthy        1726
anthracnose    1566
Name: count, dtype: int64
Imagens com classes mistas (resolvidas por maioria): 6 ['ID_BRJQ51.JPG', 'ID_I9gkRe.JPG', 'ID_nRsOlR.JPG', 'ID_r1LWke.JPG', 'ID_uKHtOl.jpeg', 'ID_yeLQCq.jpeg']

Contagens por split e classe:
label  anthracnose  cssvd  healthy
split                             
test           235    336      259
train         1096   1565     1208
val            235    336      259


## 4. Dataset e DataLoaders (com augmentation)

In [5]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

CLASSES = ["healthy", "cssvd", "anthracnose"]

class CacaoLeafDataset(Dataset):
    def __init__(self, split, transform=None):
        manifest = pd.read_csv(f"{DATA_ROOT}/manifest.csv")
        self.rows = manifest[manifest["split"] == split].reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = {c: i for i, c in enumerate(CLASSES)}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        image_path = os.path.join(DATA_ROOT, row["ImagePath"])
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.class_to_idx[row["label"]]
        return image, label

train_ds = CacaoLeafDataset("train", transform=train_transform)
val_ds = CacaoLeafDataset("val", transform=eval_transform)
test_ds = CacaoLeafDataset("test", transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

# Sanity check: inspect one batch
imgs, labels = next(iter(train_loader))
print("batch de imagens:", imgs.shape, "batch de rótulos:", labels.shape, labels[:8].tolist())


train=3869 val=830 test=830
batch de imagens: torch.Size([32, 3, 224, 224]) batch de rótulos: torch.Size([32]) [2, 2, 0, 1, 1, 1, 1, 2]


## 5. Otimização de hiperparâmetros (Optuna)

Antes do treino final (seção 6), fazemos uma busca de hiperparâmetros com **Optuna**
para escolher a taxa de aprendizado (`lr`) e o `weight_decay` do otimizador do classificador,
em vez de usar valores fixos escolhidos manualmente.

Cada *trial* do Optuna treina um classificador EfficientNet-B0 (backbone congelado, cabeça nova)
por poucas épocas (`TUNING_EPOCHS`) usando `train_loader`/`val_loader`, e retorna a melhor
acurácia de validação obtida nessas poucas épocas. Isso é uma aproximação barata do treino
completo (15 épocas) — suficiente para ranquear hiperparâmetros dentro do orçamento de tempo
de uma sessão do Kaggle, mas não perfeitamente equivalente ao desempenho do treino final.


In [6]:
import optuna
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

TUNING_EPOCHS = 4
N_TRIALS = 10


def build_tuning_model():
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    m = efficientnet_b0(weights=weights)
    for param in m.features.parameters():
        param.requires_grad = False
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
    return m.to(device)


def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    trial_model = build_tuning_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(trial_model.classifier.parameters(), lr=lr, weight_decay=weight_decay)

    best_trial_val_acc = 0.0
    for epoch in range(TUNING_EPOCHS):
        trial_model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = trial_model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        trial_model.eval()
        val_correct, val_seen = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = trial_model(imgs)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_seen += imgs.size(0)
        val_acc = val_correct / val_seen
        best_trial_val_acc = max(best_trial_val_acc, val_acc)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_trial_val_acc


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=N_TRIALS)

print(f"Melhores hiperparâmetros encontrados ({N_TRIALS} trials, {TUNING_EPOCHS} épocas cada):")
print(study.best_params)
print(f"Melhor acurácia de validação (busca): {study.best_value:.3f}")


[I 2026-09-11 18:12:20,787] A new study created in memory with name: no-name-1f0c45a0-db75-4b0f-85fa-f99c6a485206


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 130MB/s] 
[I 2026-09-11 18:27:38,818] Trial 0 finished with value: 0.7891566265060241 and parameters: {'lr': 0.00476829793570899, 'weight_decay': 0.0001246554343129267}. Best is trial 0 with value: 0.7891566265060241.
[I 2026-09-11 18:42:59,579] Trial 1 finished with value: 0.7433734939759036 and parameters: {'lr': 0.00020824706369331432, 'weight_decay': 2.3355667556809932e-06}. Best is trial 0 with value: 0.7891566265060241.
[I 2026-09-11 18:58:31,472] Trial 2 finished with value: 0.7674698795180723 and parameters: {'lr': 0.000638640477755833, 'weight_decay': 7.981122265065977e-06}. Best is trial 0 with value: 0.7891566265060241.
[I 2026-09-11 19:13:56,000] Trial 3 finished with value: 0.7927710843373494 and parameters: {'lr': 0.0014238874469165563, 'weight_decay': 0.0001255452855268151}. Best is trial 3 with value: 0.7927710843373494.
[I 2026-09-11 19:29:11,148] Trial 4 finished with value: 0.7903614457831325 and parameters: {'lr': 0.0028808

Melhores hiperparâmetros encontrados (10 trials, 4 épocas cada):
{'lr': 0.0014238874469165563, 'weight_decay': 0.0001255452855268151}
Melhor acurácia de validação (busca): 0.793


## 6. Modelo: transfer learning (EfficientNet-B0 pré-treinada)

In [7]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando device:", device)

weights = EfficientNet_B0_Weights.IMAGENET1K_V1
model = efficientnet_b0(weights=weights)

# Congela o backbone; treina apenas o classificador (rápido, adequado para dataset pequeno/médio).
for param in model.features.parameters():
    param.requires_grad = False

num_classes = len(CLASSES)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)
print(model.classifier)


Usando device: cuda
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)


## 7. Loop de treino

In [8]:
import copy
import time

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=study.best_params["lr"],
    weight_decay=study.best_params["weight_decay"],
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

NUM_EPOCHS = 15

best_val_acc = 0.0
best_state = copy.deepcopy(model.state_dict())
history = []

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # --- treino ---
    model.train()
    running_loss, running_correct, seen = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        seen += imgs.size(0)

    train_loss = running_loss / seen
    train_acc = running_correct / seen

    # --- validação ---
    model.eval()
    val_correct, val_seen = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_seen += imgs.size(0)
    val_acc = val_correct / val_seen

    scheduler.step()
    elapsed = time.time() - start
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_acc": val_acc})
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={train_loss:.4f} | train_acc={train_acc:.3f} | val_acc={val_acc:.3f} | {elapsed:.1f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

print(f"\nMelhor acurácia de validação: {best_val_acc:.3f}")
model.load_state_dict(best_state)


Epoch 1/15 | loss=0.7985 | train_acc=0.666 | val_acc=0.758 | 227.2s
Epoch 2/15 | loss=0.6474 | train_acc=0.735 | val_acc=0.771 | 229.1s
Epoch 3/15 | loss=0.6132 | train_acc=0.750 | val_acc=0.784 | 225.7s
Epoch 4/15 | loss=0.6196 | train_acc=0.748 | val_acc=0.792 | 228.1s
Epoch 5/15 | loss=0.5933 | train_acc=0.762 | val_acc=0.789 | 230.5s
Epoch 6/15 | loss=0.5596 | train_acc=0.775 | val_acc=0.800 | 231.5s
Epoch 7/15 | loss=0.5715 | train_acc=0.770 | val_acc=0.802 | 228.6s
Epoch 8/15 | loss=0.5675 | train_acc=0.768 | val_acc=0.790 | 228.0s
Epoch 9/15 | loss=0.5704 | train_acc=0.766 | val_acc=0.790 | 229.0s
Epoch 10/15 | loss=0.5732 | train_acc=0.770 | val_acc=0.798 | 228.3s
Epoch 11/15 | loss=0.5597 | train_acc=0.774 | val_acc=0.788 | 234.3s
Epoch 12/15 | loss=0.5614 | train_acc=0.768 | val_acc=0.792 | 237.1s
Epoch 13/15 | loss=0.5501 | train_acc=0.782 | val_acc=0.798 | 235.9s
Epoch 14/15 | loss=0.5538 | train_acc=0.774 | val_acc=0.793 | 232.9s
Epoch 15/15 | loss=0.5556 | train_acc=0.779

<All keys matched successfully>

## 8. Avaliação no conjunto de teste

Reporta acurácia e precisão/recall por classe, e sinaliza classes com poucas amostras de teste
(< 50, piso de PoC documentado no `design.md`) como resultado de baixa confiança.


In [9]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

report = classification_report(all_labels, all_preds, target_names=CLASSES, digits=3, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=3))

print("Matriz de confusão (linhas=real, colunas=previsto):")
print(pd.DataFrame(confusion_matrix(all_labels, all_preds), index=CLASSES, columns=CLASSES))

MIN_TEST_SAMPLES = 50
test_counts = pd.Series(all_labels).map(lambda i: CLASSES[i]).value_counts()
print("\nAvisos de confiabilidade:")
for cls in CLASSES:
    n = test_counts.get(cls, 0)
    if n < MIN_TEST_SAMPLES:
        print(f"  - AVISO: classe '{cls}' tem apenas {n} amostras de teste (< {MIN_TEST_SAMPLES}) — métrica pouco confiável.")
    else:
        print(f"  - OK: classe '{cls}' tem {n} amostras de teste.")


              precision    recall  f1-score   support

     healthy      0.718     0.826     0.768       259
       cssvd      0.793     0.765     0.779       336
 anthracnose      0.851     0.753     0.799       235

    accuracy                          0.781       830
   macro avg      0.787     0.781     0.782       830
weighted avg      0.786     0.781     0.781       830

Matriz de confusão (linhas=real, colunas=previsto):
             healthy  cssvd  anthracnose
healthy          214     35           10
cssvd             58    257           21
anthracnose       26     32          177

Avisos de confiabilidade:
  - OK: classe 'healthy' tem 259 amostras de teste.
  - OK: classe 'cssvd' tem 336 amostras de teste.
  - OK: classe 'anthracnose' tem 235 amostras de teste.


## 9. Salvar o artefato do modelo e o mapeamento de classes

In [10]:
import os
import json
import torch

assert "model" in globals(), (
    "A variável 'model' não está definida nesta sessão do kernel — "
    "isso acontece se a sessão foi reiniciada/desconectada entre o treino (seção 6) "
    "e este passo. Rode as células novamente em ordem, do início, sem interromper."
)

# Ajuste o caminho base para a pasta de saída do Kaggle
artifact_dir = "/kaggle/working/model_artifact"
os.makedirs(artifact_dir, exist_ok=True)

# 1. Salva os pesos
torch.save(model.state_dict(), f"{artifact_dir}/cacao_leaf_classifier.pt")

# 2. Salva o mapeamento de classes
label_mapping = {"classes": CLASSES, "architecture": "efficientnet_b0", "image_size": IMAGE_SIZE}
with open(f"{artifact_dir}/label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)

# 3. Salva o relatório
with open(f"{artifact_dir}/eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Artefatos salvos com sucesso em", artifact_dir)


Artefatos salvos com sucesso em /kaggle/working/model_artifact


### 9.1 Baixar os artefatos (ou salvar no Google Drive)

Baixe os três arquivos abaixo e coloque-os na pasta `models/` do projeto local
(`c:\\Fitec\\CacauFito\\models\\`) para o serviço de inferência carregar:
- `cacao_leaf_classifier.pt` — pesos do modelo treinado
- `label_mapping.json` — mapeamento índice → nome da classe (evita inconsistência entre treino e inferência)
- `eval_report.json` — métricas de avaliação (para documentar no canvas/relatório)


In [11]:
import zipfile

artifact_dir = "/kaggle/working/model_artifact"
zip_name = "/kaggle/working/cacao_leaf_model_bundle.zip"

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(f"{artifact_dir}/cacao_leaf_classifier.pt", arcname="cacao_leaf_classifier.pt")
    zipf.write(f"{artifact_dir}/label_mapping.json", arcname="label_mapping.json")
    zipf.write(f"{artifact_dir}/eval_report.json", arcname="eval_report.json")

print(f"Pacote gerado: {zip_name}")


Pacote gerado: /kaggle/working/cacao_leaf_model_bundle.zip
